# Azalyst ML Pipeline — Kaggle GPU (FIXED: Timeframe-Aware)

**Target: 1000% annual (10x capital)**

**Fix applied:** All bar-count constants (`BARS_PER_HOUR`, `BARS_PER_DAY`, `HORIZON_BARS`) are now
derived dynamically from the actual candle timeframe. The weekly loop correctly scores weekly
candles without NaN-flooding the feature matrix.

Click **Run All** and wait. Results in `/kaggle/working/results/`

In [ ]:
# ── Cell 1: Paths ─────────────────────────────────────────────────────────────
import os, sys, time, json, csv, gc, pickle, warnings
import numpy as np
import pandas as pd
import json
import time
import gc
import lightgbm as lgb
from pathlib import Path
warnings.filterwarnings('ignore')
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score

# FIXED: correct dataset path
DATA_DIR    = '/kaggle/input/datasets/dhirajsuryavanshi/binance-data-5min-300-coins-3years'
WORK_DIR    = '/kaggle/working'
FEATURE_DIR = f'{WORK_DIR}/feature_cache'
RESULTS_DIR = f'{WORK_DIR}/results'

for d in [FEATURE_DIR, RESULTS_DIR, f'{RESULTS_DIR}/models']:
    os.makedirs(d, exist_ok=True)

# Verify data exists
parquet_files = sorted(Path(DATA_DIR).rglob('*.parquet'))
if not parquet_files:
    parquet_files = sorted(Path(DATA_DIR).parent.rglob('*.parquet'))
    if parquet_files:
        DATA_DIR = str(parquet_files[0].parent)

print(f'DATA_DIR    : {DATA_DIR}')
print(f'RESULTS_DIR : {RESULTS_DIR}')
print(f'Parquet files found: {len(parquet_files)}')
for p in parquet_files[:5]:
    print(f'  {p.name}')

In [ ]:
# ── Cell 2: GPU check ─────────────────────────────────────────────────────────
import subprocess

try:
    r = subprocess.run(['nvidia-smi'], capture_output=True, timeout=5)
    HAS_GPU = r.returncode == 0
except:
    HAS_GPU = False

print(f'GPU available: {HAS_GPU}')

# Force CPU for LightGBM (Kaggle LightGBM not compiled with CUDA)
HAS_GPU = False
print('LightGBM forced to CPU mode (Kaggle build does not support CUDA)')
print('Training will still be fast — LightGBM CPU uses all cores')

try:
    import lightgbm as lgb
    print(f'LightGBM: {lgb.__version__}')
except ImportError:
    print('Installing lightgbm...')

In [ ]:
# ── Cell 3: Write azalyst_alpha_metrics.py ────────────────────────────────────

metrics_code = """
from __future__ import annotations
import warnings
from typing import Dict, List, Optional
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

ANNUAL_RETURN_TARGET = 10.0
WEEKLY_TARGET        = (1 + ANNUAL_RETURN_TARGET) ** (1 / 52) - 1
ROLLING_WINDOW_WEEKS = 4
MIN_WEEKS_TO_JUDGE   = 2
FEE_RATE             = 0.001
ROUND_TRIP_FEE       = FEE_RATE * 2

def annualised_return(weekly_returns):
    r = pd.Series(weekly_returns).dropna()
    if len(r) == 0: return 0.0
    return float((1 + r).prod() ** (52.0 / len(r)) - 1)

def sharpe_ratio(returns, periods_per_year=52):
    r = pd.Series(returns).dropna()
    if len(r) < 3 or r.std() == 0: return 0.0
    return float(r.mean() / r.std() * np.sqrt(periods_per_year))

def max_drawdown(cum_returns):
    cr = pd.Series(cum_returns).dropna()
    if len(cr) == 0: return 0.0
    peak = cr.cummax()
    return float(((cr - peak) / (1 + peak)).min())

def profit_factor(trade_pnls):
    p = pd.Series(trade_pnls)
    wins   = p[p > 0].sum()
    losses = p[p < 0].abs().sum()
    return float(wins / losses) if losses > 0 else float('inf')

def calculate_weekly_alpha(trades_df, btc_weekly_return=None):
    if trades_df is None or len(trades_df) == 0:
        return {'week_return_pct': 0.0, 'annualised_pct': 0.0,
                'sharpe': 0.0, 'excess_vs_btc_pct': None,
                'profit_factor': 0.0, 'win_rate': 0.0,
                'n_trades': 0, 'on_track': False}
    pnls        = trades_df['pnl_percent'].dropna() / 100.0
    week_return = float(pnls.mean()) if len(pnls) > 0 else 0.0
    ann         = (1 + week_return) ** 52 - 1
    excess      = (week_return - btc_weekly_return) if btc_weekly_return is not None else None
    return {
        'week_return_pct':   round(week_return * 100, 4),
        'annualised_pct':    round(ann * 100, 2),
        'sharpe':            round(sharpe_ratio(pnls, 52), 4),
        'excess_vs_btc_pct': round(excess * 100, 4) if excess is not None else None,
        'profit_factor':     round(profit_factor(pnls), 4),
        'win_rate':          round((pnls > 0).mean() * 100, 2),
        'n_trades':          len(pnls),
        'on_track':          week_return >= WEEKLY_TARGET,
    }

def should_retrain(weekly_returns_history):
    n = len(weekly_returns_history)
    if n < MIN_WEEKS_TO_JUDGE:
        return {'retrain': False, 'reason': f'Not enough weeks ({n})',
                'rolling_annual': None, 'weeks_evaluated': n}
    series    = pd.Series(weekly_returns_history)
    last_week = series.iloc[-1]
    if last_week < -0.15:
        return {'retrain': True,
                'reason': f'Catastrophic week: {last_week*100:.1f}%',
                'rolling_annual': None, 'weeks_evaluated': n}
    window      = series.iloc[-ROLLING_WINDOW_WEEKS:]
    rolling_ann = annualised_return(window)
    if rolling_ann < ANNUAL_RETURN_TARGET:
        return {'retrain': True,
                'reason': f'Rolling annualised {rolling_ann*100:.1f}% < target {ANNUAL_RETURN_TARGET*100:.0f}%',
                'rolling_annual': round(rolling_ann * 100, 2),
                'weeks_evaluated': n}
    return {'retrain': False,
            'reason': f'On track: {rolling_ann*100:.1f}% >= {ANNUAL_RETURN_TARGET*100:.0f}%',
            'rolling_annual': round(rolling_ann * 100, 2),
            'weeks_evaluated': n}

def session_report(weekly_summary_df, all_trades_df, label='Session'):
    w_rets = weekly_summary_df['week_return_pct'].dropna() / 100.0
    pnls   = all_trades_df['pnl_percent'].dropna() / 100.0
    cum    = (1 + w_rets).cumprod()
    total_ret = float(cum.iloc[-1] - 1) if len(cum) > 0 else 0.0
    ann       = annualised_return(w_rets)
    print(f'\\n{"="*60}')
    print(f'  {label.upper()} PERFORMANCE REPORT')
    print(f'{"="*60}')
    print(f'  Target        : {ANNUAL_RETURN_TARGET*100:.0f}% annual')
    print(f'  Total Return  : {total_ret*100:.2f}%')
    print(f'  Annualised    : {ann*100:.2f}%')
    print(f'  Sharpe        : {sharpe_ratio(w_rets, 52):.4f}')
    print(f'  Max Drawdown  : {max_drawdown(cum)*100:.2f}%')
    print(f'  Win Rate      : {(pnls > 0).mean()*100:.2f}%')
    print(f'  Total Trades  : {len(pnls):,}')
    wot = int((w_rets >= WEEKLY_TARGET).sum())
    print(f'  Weeks on Track: {wot} / {len(w_rets)}')
    print(f'  Alpha Achieved: {"YES" if ann >= ANNUAL_RETURN_TARGET else "NO"}')
    print(f'{"="*60}\\n')
    return {'label': label, 'total_return_pct': round(total_ret*100,2),
            'annualised_pct': round(ann*100,2),
            'alpha_achieved': ann >= ANNUAL_RETURN_TARGET}
"""

with open('/kaggle/working/azalyst_alpha_metrics.py', 'w') as f:
    f.write(metrics_code.strip())

print('azalyst_alpha_metrics.py written OK')

if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

from azalyst_alpha_metrics import (
    calculate_weekly_alpha, should_retrain, session_report,
    ROUND_TRIP_FEE, WEEKLY_TARGET, ANNUAL_RETURN_TARGET
)
print('azalyst_alpha_metrics imported OK')

In [ ]:
# ── Cell 4: All helper functions (FIXED: timeframe-aware bar counts) ──────────

# ── GLOBAL DEFAULTS (5-min training — do not change) ─────────────────────────
BARS_PER_HOUR = 12    # 60 / 5 = 12 bars per hour at 5-min
BARS_PER_DAY  = 288   # 12 * 24
HORIZON_BARS  = 48    # 4h = 48 × 5-min bars
# ─────────────────────────────────────────────────────────────────────────────

FEATURE_COLS = [
    'ret_1bar','ret_1h','ret_4h','ret_1d',
    'vol_ratio','vol_ret_1h','vol_ret_1d',
    'body_ratio','wick_top','wick_bot','candle_dir',
    'rvol_1h','rvol_4h','rvol_1d','vol_ratio_1h_1d',
    'rsi_14','rsi_6','bb_pos','bb_width',
    'vwap_dev','ctrend_12','ctrend_48','price_accel',
    'skew_1d','kurt_1d','max_ret_4h','amihud',
]
STOP_LOSS_PCT   = -1.5
TAKE_PROFIT_PCT =  4.0
TOP_QUANTILE    = 0.20


# ── FIX: dynamic bar constants per timeframe ──────────────────────────────────
def get_tf_constants(resample_str: str):
    """
    Given a pandas resample string (e.g. '5min','4h','1W','1D'),
    return (bars_per_hour, bars_per_day, horizon_bars).

    horizon_bars = "4-hour equivalent" number of bars.
    Floored at 1 so nothing breaks on very long timeframes.
    """
    s = resample_str.lower().strip()
    # Map common pandas freq strings → minutes per bar
    _map = {
        '1min': 1,  '1t': 1,
        '3min': 3,  '3t': 3,
        '5min': 5,  '5t': 5,
        '15min':15, '15t':15,
        '30min':30, '30t':30,
        '1h':  60,  '60min':60, '60t':60,
        '2h':  120,
        '4h':  240,
        '6h':  360,
        '8h':  480,
        '12h': 720,
        '1d':  1440, '1b': 1440, 'd': 1440,
        '1w':  10080,'w':  10080,'w-mon':10080,'1w-mon':10080,
    }
    mins = _map.get(s, None)
    if mins is None:
        # fallback: try to parse digit + unit
        import re
        m = re.match(r'^(\d+)([a-z]+)', s)
        if m:
            n, unit = int(m.group(1)), m.group(2)
            unit_mins = {'min':1,'t':1,'h':60,'d':1440,'w':10080}.get(unit,1)
            mins = n * unit_mins
        else:
            mins = 5  # safe default

    bph = max(1, 60   // mins)   # bars per hour
    bpd = max(1, 1440 // mins)   # bars per day
    hor = max(1, 240  // mins)   # 4h-equivalent horizon
    return bph, bpd, hor


def _rsi(s, n):
    d  = s.diff()
    g  = d.clip(lower=0).ewm(alpha=1/n, adjust=False).mean()
    ls = (-d).clip(lower=0).ewm(alpha=1/n, adjust=False).mean()
    return 100 - 100 / (1 + g / ls.replace(0, np.nan))


def load_ohlcv(path):
    """Load one parquet file → clean OHLCV DataFrame with UTC DatetimeIndex."""
    try:
        df = pd.read_parquet(path)
        df.columns = [c.lower() for c in df.columns]
        ts_col = next((c for c in df.columns
                       if c in ('timestamp', 'time', 'open_time')), None)
        if ts_col:
            col = df[ts_col]
            if pd.api.types.is_integer_dtype(col):
                df.index = pd.to_datetime(col, unit='ms', utc=True)
            else:
                df.index = pd.to_datetime(col, utc=True)
            df = df.drop(columns=[ts_col])
        elif isinstance(df.index, pd.DatetimeIndex):
            df.index = (df.index.tz_localize('UTC')
                        if df.index.tz is None
                        else df.index.tz_convert('UTC'))
        else:
            df.index = pd.to_datetime(df.index, utc=True)
        df = df.sort_index()
        for c in ['open', 'high', 'low', 'close', 'volume']:
            if c not in df.columns:
                return None
        return (df[['open','high','low','close','volume']]
                .apply(pd.to_numeric, errors='coerce')
                .dropna())
    except Exception:
        return None


# ── FIX: build_features now accepts timeframe ─────────────────────────────────
def build_features(df, resample='5min'):
    """
    Compute 27 ML features.  Shift +1 bar — zero lookahead.
    All rolling windows are derived from resample so the features
    are semantically identical regardless of candle size.
    """
    bph, bpd, hor = get_tf_constants(resample)

    c = df['close']; o = df['open']
    h = df['high'];  l = df['low'];  v = df['volume']
    f = pd.DataFrame(index=df.index)
    lr = np.log(c / c.shift(1))

    f['ret_1bar'] = lr
    f['ret_1h']   = np.log(c / c.shift(bph))
    f['ret_4h']   = np.log(c / c.shift(bph * 4))
    f['ret_1d']   = np.log(c / c.shift(bpd))

    av = v.rolling(bpd, min_periods=max(2, bph)).mean()
    f['vol_ratio']   = v / av.replace(0, np.nan)
    f['vol_ret_1h']  = np.log(v / v.shift(bph).replace(0, np.nan))
    f['vol_ret_1d']  = np.log(v / v.shift(bpd).replace(0, np.nan))

    rng = (h - l).replace(0, np.nan)
    f['body_ratio'] = (c - o).abs() / rng
    f['wick_top']   = (h - c.clip(lower=o)) / rng
    f['wick_bot']   = (c.clip(upper=o) - l) / rng
    f['candle_dir'] = np.sign(c - o)

    f['rvol_1h']  = lr.rolling(bph,     min_periods=max(2, bph // 2)).std()
    f['rvol_4h']  = lr.rolling(bph * 4, min_periods=max(2, bph)).std()
    f['rvol_1d']  = lr.rolling(bpd,     min_periods=max(2, bph)).std()
    f['vol_ratio_1h_1d'] = f['rvol_1h'] / f['rvol_1d'].replace(0, np.nan)

    f['rsi_14'] = _rsi(c, 14) / 100.0
    f['rsi_6']  = _rsi(c,  6) / 100.0

    ma  = c.rolling(20, min_periods=10).mean()
    std = c.rolling(20, min_periods=10).std(ddof=0)
    bw  = (4 * std).replace(0, np.nan)
    f['bb_pos']   = ((c - (ma - 2 * std)) / bw).clip(0, 1)
    f['bb_width'] = bw / ma.replace(0, np.nan)

    tp   = (h + l + c) / 3
    vwap = ((tp * v).rolling(bpd, min_periods=max(2, bph)).sum() /
             v.rolling(bpd, min_periods=max(2, bph)).sum().replace(0, np.nan))
    f['vwap_dev']    = (c - vwap) / c.replace(0, np.nan)

    s = np.sign(lr)
    # ctrend_12 / 48 keep absolute bar counts (12 & 48) at training timeframe;
    # at longer TFs we scale them proportionally so they mean the same duration.
    ct12 = max(2, bph)           # ~1h of bars
    ct48 = max(2, bph * 4)       # ~4h of bars
    f['ctrend_12']   = s.rolling(ct12, min_periods=max(2, ct12 // 2)).sum()
    f['ctrend_48']   = s.rolling(ct48, min_periods=max(2, ct48 // 2)).sum()

    m1 = c.pct_change(bph)
    f['price_accel'] = m1 - m1.shift(bph)

    f['skew_1d']    = lr.rolling(bpd, min_periods=max(4, bph)).skew()
    f['kurt_1d']    = lr.rolling(bpd, min_periods=max(4, bph)).kurt()
    f['max_ret_4h'] = lr.rolling(bph * 4, min_periods=max(2, bph)).max()
    f['amihud']     = (lr.abs() / v.replace(0, np.nan)).rolling(
                       bpd, min_periods=max(2, bph)).mean()

    return f.replace([np.inf, -np.inf], np.nan).shift(1)  # +1 bar no-lookahead


# ── FIX: build_panel passes resample down to build_features ───────────────────
def build_panel(data_dir, date_from, date_to, resample='4h', verbose=True):
    """
    Load all symbols in date window, compute features, resample.
    resample controls BOTH the output candle size AND the feature
    rolling-window lengths — they are now consistent.
    """
    bph, bpd, hor = get_tf_constants(resample)
    min_bars = bpd * 3   # require 3 days worth of bars after resampling

    files  = sorted(Path(data_dir).glob('*.parquet'))
    frames = []
    for i, f in enumerate(files, 1):
        try:
            df = load_ohlcv(f)
            if df is None or len(df) < BARS_PER_DAY * 3:  # raw 5-min check
                continue
            df = df[(df.index >= date_from) & (df.index < date_to)]
            if len(df) < BARS_PER_DAY * 3:
                continue

            # Resample raw 5-min OHLCV to target TF first, then build features
            if resample not in ('5min', '5T', '5t'):
                agg = {
                    'open':  'first',
                    'high':  'max',
                    'low':   'min',
                    'close': 'last',
                    'volume':'sum',
                }
                df = df.resample(resample, label='left', closed='left').agg(agg).dropna()
                if len(df) < max(5, min_bars):
                    continue

            feat = build_features(df, resample=resample)

            # Forward return label scaled to the target horizon
            feat['future_ret_4h'] = np.log(
                df['close'].shift(-hor) / df['close'])
            feat['symbol'] = f.stem

            avail_cols = [c for c in FEATURE_COLS if c in feat.columns]

            # For 5-min base data we still resample to 4H for the panel;
            # for longer TFs the data is already at target resolution.
            if resample in ('5min', '5T', '5t'):
                feat_rs = feat.resample('4h').last().dropna(
                    subset=avail_cols, how='all')
            else:
                feat_rs = feat.dropna(subset=avail_cols, how='all')

            if len(feat_rs) < 5:
                continue
            frames.append(feat_rs)
        except Exception:
            continue
        if verbose and i % 50 == 0:
            print(f'  {i}/{len(files)} scanned, {len(frames)} valid')

    if not frames:
        return pd.DataFrame()
    result = pd.concat(frames).sort_index()
    if verbose:
        syms = result['symbol'].nunique() if 'symbol' in result.columns else '?'
        print(f'  Panel: {len(result):,} rows, {syms} symbols  [resample={resample}]')
    return result


# ── Cross-sectional helpers (fixed: transform instead of .loc) ────────────────
def add_alpha_labels(df):
    """Cross-sectional alpha label: 1 = outperforms universe median."""
    df = df.copy()
    median_by_ts = df.groupby(level=0)['future_ret_4h'].transform('median')
    df['alpha_label'] = np.where(
        df['future_ret_4h'].notna(),
        (df['future_ret_4h'] > median_by_ts).astype(float),
        np.nan
    )
    return df


def cs_rank(df, cols):
    """Cross-sectional percentile rank at each timestamp."""
    avail = [c for c in cols if c in df.columns]
    ranked_parts = []
    for ts, group in df.groupby(level=0):
        if len(group) < 2:
            ranked_parts.append(group)
            continue
        g = group.copy()
        g[avail] = group[avail].rank(pct=True)
        ranked_parts.append(g)
    return pd.concat(ranked_parts)


def lgbm_params(use_gpu=False):
    p = {
        'objective': 'binary', 'metric': 'auc',
        'n_estimators': 1000, 'learning_rate': 0.03,
        'num_leaves': 127 if use_gpu else 63,
        'max_bin': 255, 'min_child_samples': 20,
        'subsample': 0.8, 'subsample_freq': 1,
        'colsample_bytree': 0.8, 'class_weight': 'balanced',
        'random_state': 42, 'verbose': -1,
    }
    if use_gpu:
        p['device'] = 'cuda'
        p['gpu_use_dp'] = False
        p['n_jobs'] = 1
    else:
        p['n_jobs'] = -1
    return p


def train_lgbm(X, y, feature_cols, use_gpu=False, label=''):
    """Train LightGBM with purged time-series CV."""
    scaler = StandardScaler()
    Xs     = scaler.fit_transform(X)
    print(f'  Training [{label}]: {len(X):,} samples, {X.shape[1]} features')
    aucs   = []
    for fold, (tr, val) in enumerate(
            TimeSeriesSplit(n_splits=3, gap=48).split(Xs), 1):
        if len(np.unique(y[val])) < 2:
            continue
        m = lgb.LGBMClassifier(**lgbm_params(use_gpu))
        m.fit(Xs[tr], y[tr],
              eval_set=[(Xs[val], y[val])],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(period=-1)])
        try:
            auc = roc_auc_score(y[val], m.predict_proba(Xs[val])[:, 1])
            aucs.append(auc)
            print(f'    Fold {fold}: AUC={auc:.4f}')
        except Exception:
            pass
    mean_auc = float(np.mean(aucs)) if aucs else 0.0
    print(f'  CV Mean AUC: {mean_auc:.4f}')
    final = lgb.LGBMClassifier(**lgbm_params(use_gpu))
    split = int(len(Xs) * 0.9)
    final.fit(Xs[:split], y[:split],
              eval_set=[(Xs[split:], y[split:])],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(period=-1)])
    importance = pd.Series(
        getattr(final, 'feature_importances_', np.zeros(len(feature_cols))),
        index=feature_cols, name='importance'
    ).sort_values(ascending=False)
    return final, scaler, importance, mean_auc


def save_model(model, scaler, feature_cols, path, meta=None):
    payload = {'model': model, 'scaler': scaler, 'feature_cols': feature_cols}
    if meta:
        payload.update(meta)
    with open(path, 'wb') as fh:
        pickle.dump(payload, fh)


def load_model(path):
    with open(path, 'rb') as fh:
        obj = pickle.load(fh)
    return obj['model'], obj['scaler'], obj['feature_cols']


import builtins
builtins.add_alpha_labels = add_alpha_labels
print('All helper functions defined OK')
print('get_tf_constants test:')
for tf in ['5min','4h','1D','1W']:
    bph, bpd, hor = get_tf_constants(tf)
    print(f'  {tf:8s}  bph={bph:4d}  bpd={bpd:5d}  horizon={hor}')

In [ ]:
# ── Cell 5: Discover date range ───────────────────────────────────────────────
print('Scanning date range...')
starts, ends = [], []
files = sorted(Path(DATA_DIR).glob('*.parquet'))[:80]
print(f'Checking {len(files)} files...')

for f in files:
    try:
        df = load_ohlcv(f)
        if df is not None and len(df) > BARS_PER_DAY * 30:
            starts.append(df.index.min())
            ends.append(df.index.max())
    except Exception:
        pass

if not starts:
    raise RuntimeError(
        f'No valid parquet files in {DATA_DIR}\n'
        f'Files found: {[f.name for f in files[:5]]}\n'
        f'Try running the path-finder cell first.'
    )

global_start = min(starts)
global_end   = max(ends)
year1_end    = global_start + pd.Timedelta(days=365)
year2_end    = year1_end    + pd.Timedelta(days=365)
year3_end    = global_end

print(f'Global start : {global_start.date()}')
print(f'Global end   : {global_end.date()}')
print(f'Year 1       : {global_start.date()} → {year1_end.date()}')
print(f'Year 2       : {year1_end.date()} → {year2_end.date()}')
print(f'Year 3       : {year2_end.date()} → {year3_end.date()}')
print(f'Symbols      : {len(starts)}')

date_config = {
    'global_start': global_start.isoformat(),
    'global_end':   global_end.isoformat(),
    'year1_end':    year1_end.isoformat(),
    'year2_end':    year2_end.isoformat(),
    'year3_end':    year3_end.isoformat(),
}
with open(f'{RESULTS_DIR}/date_config.json', 'w') as fh:
    json.dump(date_config, fh, indent=2)
print('date_config.json saved')

In [ ]:
# ── Cell 6: YEAR 1 TRAINING ───────────────────────────────────────────────────
# Trained on 5-min data resampled to 4H (same as original).
# build_panel now correctly propagates resample='5min' to build_features.

print('=' * 60)
print('  YEAR 1 TRAINING')
print('=' * 60)
t0 = time.time()

print('Loading Year 1 data...')
df_y1 = build_panel(DATA_DIR, global_start, year1_end,
                    resample='5min', verbose=True)

if df_y1.empty:
    raise RuntimeError('No Year 1 data loaded. Check DATA_DIR.')

print(f'Year 1 panel: {len(df_y1):,} rows')

df_y1    = add_alpha_labels(df_y1)
avail_y1 = [c for c in FEATURE_COLS if c in df_y1.columns]
valid_y1 = df_y1.dropna(subset=avail_y1 + ['alpha_label'])
valid_y1 = cs_rank(valid_y1, avail_y1)
print(f'Valid labelled rows : {len(valid_y1):,}')
print(f'Alpha rate          : {valid_y1["alpha_label"].mean()*100:.1f}%  (expected ~50%)')

X_y1 = valid_y1[avail_y1].values.astype(np.float32)
y_y1 = valid_y1['alpha_label'].values.astype(int)

model_y1, scaler_y1, imp_y1, auc_y1 = train_lgbm(
    X_y1, y_y1, avail_y1, use_gpu=HAS_GPU, label='Year1'
)

y1_model_path = f'{RESULTS_DIR}/models/model_year1.pkl'
save_model(model_y1, scaler_y1, avail_y1, y1_model_path,
           {'cv_auc': round(auc_y1, 4), 'n_train_rows': len(X_y1)})
imp_y1.to_csv(f'{RESULTS_DIR}/feature_importance_year1.csv')

print(f'\nTop 10 features:')
for feat, val in imp_y1.head(10).items():
    print(f'  {feat:<25}  {val:>8.1f}')

train_summary = {
    'year1_end':      year1_end.isoformat(),
    'n_symbols':      int(df_y1['symbol'].nunique()),
    'n_train_rows':   int(len(X_y1)),
    'n_features':     len(avail_y1),
    'alpha_rate_pct': round(float(y_y1.mean()) * 100, 2),
    'cv_auc':         round(auc_y1, 4),
    'elapsed_min':    round((time.time() - t0) / 60, 2),
    'gpu_used':       HAS_GPU,
}
with open(f'{RESULTS_DIR}/train_summary.json', 'w') as fh:
    json.dump(train_summary, fh, indent=2)

del df_y1, valid_y1
gc.collect()
print(f'\nYear 1 training done in {(time.time()-t0)/60:.1f} min')
print(f'Model saved: {y1_model_path}')

In [ ]:
# ── Cell 7: Weekly loop function (FIXED) ──────────────────────────────────────
#
# Key changes vs original:
#  1. build_panel called with resample='5min' so features are computed on
#     the correct scale — the 4H resampling happens INSIDE build_panel.
#  2. HORIZON_BARS for trade simulation is derived per-symbol from the
#     actual raw 5-min data, not from a global constant that breaks on
#     other timeframes.
#  3. Retrain also uses resample='5min' for consistency with Year 1.

def run_year_loop(year_label, year_start, year_end,
                  seed_model, seed_scaler, seed_feature_cols):

    model         = seed_model
    scaler        = seed_scaler
    feat_cols     = list(seed_feature_cols)
    current_model = f'model_{year_label.lower()}_seed'

    all_trades     = []
    weekly_summary = []
    weekly_returns = []
    retrain_count  = 0

    # HORIZON for trade sim stays fixed at 5-min equivalent of 4h = 48 bars
    SIM_HORIZON = HORIZON_BARS  # 48 × 5-min bars

    week_starts = pd.date_range(
        year_start,
        year_end - pd.Timedelta(weeks=1),
        freq='W-MON',
        tz='UTC'
    )

    print(f'\n{"="*60}')
    print(f'  {year_label} — {len(week_starts)} weeks')
    print(f'  {year_start.date()} → {year_end.date()}')
    print(f'{"="*60}')

    for wk_num, week_start in enumerate(week_starts, 1):

        week_end = week_start + pd.Timedelta(weeks=1)
        if week_end > year_end + pd.Timedelta(days=3):
            break

        t_wk = time.time()

        print(f'\n  Week {wk_num}/{len(week_starts)}: '
              f'{week_start.date()} → {week_end.date()}')

        # ── Load data (5-min → 4H panel, features correctly scaled) ──────
        week_df = build_panel(DATA_DIR, week_start, week_end,
                              resample='5min', verbose=False)

        if week_df.empty:
            print('    [SKIP] No data')
            weekly_returns.append(0.0)
            weekly_summary.append({
                'year': year_label, 'week_num': wk_num,
                'week_start': week_start.date().isoformat(),
                'week_end': week_end.date().isoformat(),
                'week_return_pct': 0.0, 'annualised_pct': 0.0,
                'sharpe': 0.0, 'profit_factor': 0.0,
                'win_rate': 0.0, 'n_trades': 0, 'on_track': False,
                'rolling_annual': None, 'retrained': False,
                'retrain_count_so_far': retrain_count,
                'model': current_model,
            })
            continue

        week_ranked = cs_rank(week_df, feat_cols)
        avail = [c for c in feat_cols if c in week_ranked.columns]

        if len(avail) == 0:
            print('    No usable features')
            continue

        week_ranked['prob']   = np.nan
        week_ranked['signal'] = 'HOLD'

        # ── Generate signals ──────────────────────────────────────────────
        for ts, group in week_ranked.groupby(level=0):
            valid_rows = group.dropna(subset=avail)
            if len(valid_rows) < 5:
                continue
            try:
                Xs    = scaler.transform(valid_rows[avail].values.astype(np.float32))
                probs = model.predict_proba(Xs)[:, 1]
                prob_series = pd.Series(probs, index=valid_rows.index)
                week_ranked.loc[prob_series.index, 'prob'] = prob_series
            except Exception:
                continue
            n_long     = max(1, int(len(valid_rows) * TOP_QUANTILE))
            sorted_idx = valid_rows.index[np.argsort(probs)]
            week_ranked.loc[sorted_idx[-n_long:], 'signal'] = 'BUY'
            week_ranked.loc[sorted_idx[:n_long],  'signal'] = 'SELL'

        n_sigs = int((week_ranked['signal'] != 'HOLD').sum())
        print(f'    Signals: {n_sigs}')

        # ── Simulate trades (always on raw 5-min bars) ────────────────────
        week_trades = []
        sig_rows    = week_ranked[week_ranked['signal'].isin(['BUY', 'SELL'])]

        for ts, row in sig_rows.iterrows():
            sym        = str(row.get('symbol', ''))
            signal     = str(row['signal'])
            prob       = float(row.get('prob', 0.5))
            ohlcv_path = Path(DATA_DIR) / f'{sym}.parquet'
            if not ohlcv_path.exists():
                continue
            ohlcv = load_ohlcv(ohlcv_path)
            if ohlcv is None:
                continue
            # Use raw 5-min bars for simulation regardless of signal TF
            future = ohlcv[ohlcv.index > ts].head(SIM_HORIZON + 10)
            if len(future) < 2:
                continue
            entry_price = float(future.iloc[0]['open'])
            if entry_price <= 0:
                continue

            if signal == 'BUY':
                sl_p = entry_price * (1 + STOP_LOSS_PCT   / 100)
                tp_p = entry_price * (1 + TAKE_PROFIT_PCT / 100)
            else:
                sl_p = entry_price * (1 - STOP_LOSS_PCT   / 100)
                tp_p = entry_price * (1 - TAKE_PROFIT_PCT / 100)

            exit_price  = None
            exit_reason = 'horizon'

            for _, bar in future.iloc[1:SIM_HORIZON + 1].iterrows():
                lo = float(bar['low'])
                hi = float(bar['high'])
                if signal == 'BUY':
                    if lo <= sl_p:
                        exit_price  = sl_p
                        exit_reason = 'stop_loss'
                        break
                    if hi >= tp_p:
                        exit_price  = tp_p
                        exit_reason = 'take_profit'
                        break
                else:
                    if hi >= sl_p:
                        exit_price  = sl_p
                        exit_reason = 'stop_loss'
                        break
                    if lo <= tp_p:
                        exit_price  = tp_p
                        exit_reason = 'take_profit'
                        break

            if exit_price is None:
                exit_price = float(
                    future.iloc[min(SIM_HORIZON, len(future)-1)]['close'])

            raw_ret = exit_price / entry_price - 1
            if signal == 'SELL':
                raw_ret = -raw_ret
            pnl_pct = (raw_ret - ROUND_TRIP_FEE) * 100

            week_trades.append({
                'year':        year_label,
                'week_num':    wk_num,
                'signal_time': ts.isoformat(),
                'symbol':      sym,
                'signal':      signal,
                'probability': round(prob, 4),
                'entry_price': round(entry_price, 8),
                'exit_price':  round(exit_price, 8),
                'pnl_percent': round(pnl_pct, 4),
                'result':      'WIN' if pnl_pct > 0 else 'LOSS',
                'exit_reason': exit_reason,
            })

        all_trades.extend(week_trades)
        trades_df = pd.DataFrame(week_trades) if week_trades else None

        # ── Evaluate performance ──────────────────────────────────────────
        alpha = calculate_weekly_alpha(trades_df)
        weekly_returns.append(alpha['week_return_pct'] / 100)
        print(
            f'    Return: {alpha["week_return_pct"]:+.2f}%  '
            f'Ann: {alpha["annualised_pct"]:.0f}%  '
            f'WR: {alpha["win_rate"]:.0f}%  '
            f'Trades: {alpha["n_trades"]}'
        )

        # ── Retrain decision ──────────────────────────────────────────────
        decision  = should_retrain(weekly_returns)
        retrained = False

        if decision['retrain']:
            print(f'    RETRAIN: {decision["reason"]}')
            retrain_count += 1

            # Retrain on all history up to now using 5-min data (same as Year 1)
            train_df = build_panel(
                DATA_DIR, global_start, week_end,
                resample='5min', verbose=False
            )

            if not train_df.empty:
                train_df = add_alpha_labels(train_df)
                avail2   = [c for c in feat_cols if c in train_df.columns]
                valid2   = train_df.dropna(subset=avail2 + ['alpha_label'])
                valid2   = cs_rank(valid2, avail2)

                if len(valid2) > 200:
                    X2 = valid2[avail2].values.astype(np.float32)
                    y2 = valid2['alpha_label'].values.astype(int)

                    new_model, new_scaler, imp, cv_auc = train_lgbm(
                        X2, y2, avail2,
                        use_gpu=HAS_GPU,
                        label=f'{year_label}_wk{wk_num}'
                    )

                    model     = new_model
                    scaler    = new_scaler
                    feat_cols = avail2

                    fname         = f'model_{year_label.lower()}_week{wk_num:03d}.pkl'
                    current_model = fname
                    save_model(
                        model, scaler, feat_cols,
                        f'{RESULTS_DIR}/models/{fname}',
                        {'cv_auc': round(cv_auc, 4), 'week': wk_num,
                         'retrain_reason': decision['reason']}
                    )
                    imp.to_csv(
                        f'{RESULTS_DIR}/feature_importance_'
                        f'{year_label.lower()}_week{wk_num:03d}.csv'
                    )
                    print(f'    Retrained → {fname}  AUC={cv_auc:.4f}')
                    retrained = True

                del train_df
                gc.collect()
        else:
            print(f'    Alpha OK: {decision["reason"]}')

        weekly_summary.append({
            'year':                  year_label,
            'week_num':              wk_num,
            'week_start':            week_start.date().isoformat(),
            'week_end':              week_end.date().isoformat(),
            'week_return_pct':       alpha['week_return_pct'],
            'annualised_pct':        alpha['annualised_pct'],
            'sharpe':                alpha['sharpe'],
            'profit_factor':         alpha['profit_factor'],
            'win_rate':              alpha['win_rate'],
            'n_trades':              alpha['n_trades'],
            'on_track':              alpha['on_track'],
            'rolling_annual':        decision.get('rolling_annual'),
            'retrained':             retrained,
            'retrain_count_so_far':  retrain_count,
            'model':                 current_model,
        })

        del week_df, week_ranked
        gc.collect()

        print(
            f'    Week done in {time.time()-t_wk:.1f}s'
            f' | Total retrains: {retrain_count}'
        )

    wk_df = pd.DataFrame(weekly_summary)
    tr_df = pd.DataFrame(all_trades)

    wk_df.to_csv(f'{RESULTS_DIR}/weekly_summary_{year_label.lower()}.csv', index=False)
    tr_df.to_csv(f'{RESULTS_DIR}/all_trades_{year_label.lower()}.csv',     index=False)

    print(f'\n  {year_label} complete. Retrains: {retrain_count}')
    return wk_df, tr_df, model, scaler, feat_cols


print('Weekly loop function defined OK')

In [ ]:
# ── Cell 8: YEAR 2 LOOP ───────────────────────────────────────────────────────
wk2, tr2, model_y2, scaler_y2, feats_y2 = run_year_loop(
    'Year2',
    year1_end,
    year2_end,
    model_y1,
    scaler_y1,
    avail_y1,
)

if not wk2.empty and not tr2.empty:
    session_report(wk2, tr2, label='Year 2')

In [ ]:
# ── Cell 9: YEAR 3 LOOP ───────────────────────────────────────────────────────
wk3, tr3, model_y3, scaler_y3, feats_y3 = run_year_loop(
    'Year3',
    year2_end,
    year3_end,
    model_y2,
    scaler_y2,
    feats_y2,
)

if not wk3.empty and not tr3.empty:
    session_report(wk3, tr3, label='Year 3')

In [ ]:
# ── Cell 10: COMBINED REPORT ──────────────────────────────────────────────────
frames_wk = [df for df in [wk2, wk3] if not df.empty]
frames_tr = [df for df in [tr2, tr3] if not df.empty]

all_wk = pd.concat(frames_wk, ignore_index=True) if frames_wk else pd.DataFrame()
all_tr = pd.concat(frames_tr, ignore_index=True) if frames_tr else pd.DataFrame()

if not all_wk.empty:
    all_wk.to_csv(f'{RESULTS_DIR}/weekly_summary_all.csv', index=False)
if not all_tr.empty:
    all_tr.to_csv(f'{RESULTS_DIR}/all_trades_all.csv', index=False)

combined_rpt = {}
if not all_wk.empty and not all_tr.empty:
    combined_rpt = session_report(all_wk, all_tr, label='Combined Year2+3')

alpha_report = {
    'target':             '1000% annual (10x capital)',
    'total_weeks':        len(all_wk),
    'total_trades':       len(all_tr),
    'year2_mean_ann_pct': round(wk2['annualised_pct'].mean(), 2) if not wk2.empty else None,
    'year3_mean_ann_pct': round(wk3['annualised_pct'].mean(), 2) if not wk3.empty else None,
    'year2_retrains':     int(wk2['retrained'].sum()) if not wk2.empty else 0,
    'year3_retrains':     int(wk3['retrained'].sum()) if not wk3.empty else 0,
    'overall_win_rate':   round(float((all_tr['pnl_percent'] > 0).mean() * 100), 2)
                          if not all_tr.empty else 0,
    'weeks_on_track':     int(all_wk['on_track'].sum()) if not all_wk.empty else 0,
    'alpha_achieved':     combined_rpt.get('alpha_achieved', False),
}
with open(f'{RESULTS_DIR}/alpha_report.json', 'w') as fh:
    json.dump(alpha_report, fh, indent=2)

print('\nALPHA REPORT:')
for k, v in alpha_report.items():
    print(f'  {k:<30}  {v}')

print('\nALL OUTPUT FILES:')
for f in sorted(Path(RESULTS_DIR).rglob('*')):
    if f.is_file():
        kb = f.stat().st_size // 1024
        print(f'  {str(f.relative_to(RESULTS_DIR)):<50}  {kb} KB')

In [ ]:
# ── Cell 11: ZIP for download ─────────────────────────────────────────────────
import shutil

zip_base = f'{WORK_DIR}/azalyst_results'
shutil.make_archive(zip_base, 'zip', RESULTS_DIR)
zip_path = zip_base + '.zip'

size_mb = Path(zip_path).stat().st_size / 1e6
print(f'azalyst_results.zip  —  {size_mb:.1f} MB')
print(f'Location: {zip_path}')
print()
print('DONE — go to Kaggle Output tab and download azalyst_results.zip')
print()
print('Files to send to Claude:')
print('  1. alpha_report.json')
print('  2. weekly_summary_all.csv')
print('  3. all_trades_all.csv')
print('  4. feature_importance_year1.csv')
print('  5. feature_importance_year*_week*.csv  (each retrain)')